# 03 — Bring Your Own Optimizer

Same Tubulin objective as tutorial 2, but the search algorithm is swappable.
The point: VQGAE's latent for inverse design is just a 4096-dim
**integer histogram with `sum ≤ 51`**. Anything that searches that space
works — Genetic Algorithms, Bayesian Optimization (TPE / SMAC / discrete BO),
simulated annealing, RL, …

Three drop-in implementations of the same scoring loop:

| Path | Search algorithm | Library | Best for |
|------|-------------------|---------|----------|
| A    | GA (single-objective) | PyGAD | Cheap, reproducible, used in the paper |
| B    | TPE (discrete Bayesian opt) | Optuna | Sample-efficient when scoring is expensive |
| C    | Random search | numpy | Sanity baseline |

> **What about continuous BO (BoTorch)?**
> The decoder operates on integer codebook indices, so a GP surrogate over a
> continuous box doesn't directly produce decodable inputs. The encoder *does*
> return a 512-dim continuous `feature_vector` per molecule, but the public
> `VQGAE.decode` API does not currently consume it. See the last section of
> this notebook for the small API addition that would unlock continuous BO.


## Shared setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

from VQGAE import (
    VQGAE, OrderingNetwork,
    decode_population, tanimoto_kernel, filter_molecule,
)

DATA = "../data/tubulin"
X = np.load(f"{DATA}/tubulin_qsar_class_train_data_vqgae.npz")["x"]
Y = np.load(f"{DATA}/tubulin_qsar_class_train_data_vqgae.npz")["y"]
with open(f"{DATA}/rf_class_train_tubulin.pickle", "rb") as fh:
    rf_model = pickle.load(fh)

REPO = "tagirshin/VQGAE"
BATCH = 500
ordering_model = OrderingNetwork.load_from_checkpoint(
    hf_hub_download(REPO, "ordering_network.ckpt"), batch_size=BATCH, map_location="cpu"
).eval()
vqgae_dec = VQGAE.load_from_checkpoint(
    hf_hub_download(REPO, "vqgae.ckpt"), task="decode", batch_size=50, map_location="cpu"
).eval()

## Single building block: a `score(counts) -> array` function

Every black-box optimizer expects exactly this contract.  The package's
`tanimoto_kernel` plus your QSAR model is all you need.


In [ ]:
def score(counts: np.ndarray) -> np.ndarray:
    """0.5·RF + 0.3·dissimilarity − size_penalty. Higher is better."""
    counts = np.atleast_2d(counts)
    rf = rf_model.predict_proba(counts)[:, 1]
    size_pen = np.where(counts.sum(-1) < 18, -1.0, 0.0)
    dissim = 1 - tanimoto_kernel(counts, X).max(-1)
    dissim += np.where(dissim == 0, -5, 0)
    return 0.5 * rf + 0.3 * dissim + size_pen

---
## Path A — Genetic Algorithm (PyGAD baseline, used in the paper)

In [ ]:
import pygad

def fitness_batch(_ga, solutions, _idx):
    return score(np.array(solutions)).tolist()

NUM_GENS_GA = 5     # bump to 30 for the full paper-style run
ga = pygad.GA(
    fitness_func=fitness_batch,
    initial_population=X,
    num_genes=X.shape[-1],
    fitness_batch_size=BATCH,
    num_generations=NUM_GENS_GA,
    num_parents_mating=190,
    parent_selection_type="rws",
    crossover_type="single_point",
    mutation_type="adaptive",
    mutation_percent_genes=[10, 5],
    save_solutions=True,
    keep_elitism=0,
    keep_parents=120,
    suppress_warnings=True,
    random_seed=42,
    gene_type=int,
)
ga.run()
ga_solutions = list({tuple(s) for s in ga.solutions})
ga_scores = score(np.array(ga_solutions))
print(f"GA: {len(ga_solutions)} unique solutions, top score {ga_scores.max():.3f}")

---
## Path B — Optuna (Tree-structured Parzen Estimator)

TPE is sample-efficient discrete BO. It shines when each evaluation of `score`
is expensive — e.g., docking, FEP, wet-lab assays — but the same template
works here.

Key trick: don't let Optuna pick 4096 independent integers (that ignores the
`sum ≤ 51` constraint). Instead, sample
**(target heavy-atom count, number of distinct fragments)** and a fragment
selection from a popularity-prior, then a multinomial split. This keeps every
candidate in-distribution.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_FRAGS = X.shape[-1]
MAX_ATOMS = 51

# popularity prior over the training set
top_k_frags = np.argsort(-(X > 0).mean(0))[:200]


def sample_candidate(trial: optuna.Trial) -> np.ndarray:
    counts = np.zeros(N_FRAGS, dtype=np.int64)
    target_atoms = trial.suggest_int("target_atoms", 18, MAX_ATOMS)
    n_distinct = trial.suggest_int("n_distinct", 5, 20)
    rng = np.random.default_rng(trial.number)
    chosen = rng.choice(top_k_frags, size=n_distinct, replace=False)
    splits = rng.multinomial(target_atoms, [1 / n_distinct] * n_distinct)
    for f, k in zip(chosen, splits):
        counts[f] = k
    return counts


def objective(trial: optuna.Trial) -> float:
    counts = sample_candidate(trial)
    trial.set_user_attr("counts", counts.tolist())
    return float(score(counts).item())


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=50),
)
study.optimize(objective, n_trials=400, show_progress_bar=True)

bo_solutions = [np.array(t.user_attrs["counts"]) for t in study.trials if "counts" in t.user_attrs]
bo_scores = np.array([t.value for t in study.trials if t.value is not None])
print(f"Optuna TPE: {len(bo_solutions)} trials, top score {bo_scores.max():.3f}")

---
## Path C — Random search (sanity baseline)

Sample fragment counts directly from the training distribution. If smart
optimizers don't beat this, something is wrong.


In [ ]:
rng = np.random.default_rng(42)
N_RAND = 2000
rand_idx = rng.integers(0, X.shape[0], size=N_RAND)
rand_solutions = X[rand_idx].copy()
flip = rng.integers(0, X.shape[1], size=N_RAND)
rand_solutions[np.arange(N_RAND), flip] = np.maximum(
    0, rand_solutions[np.arange(N_RAND), flip] + rng.choice([-1, 1], size=N_RAND)
)
rand_scores = score(rand_solutions)
print(f"Random: top score {rand_scores.max():.3f}")

---
## Compare and decode the top-50 from each path

In [ ]:
def top_k(solutions, scores, k=50):
    idx = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(solutions)[idx], np.asarray(scores)[idx]

ga_top, ga_top_sc   = top_k(ga_solutions, ga_scores)
bo_top, bo_top_sc   = top_k(bo_solutions, bo_scores)
rand_top, rand_top_sc = top_k(rand_solutions, rand_scores)

print("Top-50 mean fitness:")
print(f"  GA:     {ga_top_sc.mean():.3f}")
print(f"  Optuna: {bo_top_sc.mean():.3f}")
print(f"  Random: {rand_top_sc.mean():.3f}")

In [ ]:
def decode_and_filter(top_counts):
    mols, validity, _ = decode_population(
        top_counts.astype(np.int64), vqgae_dec, ordering_model, batch_size=50
    )
    return [m for m, ok in zip(mols, validity) if ok and filter_molecule(m)]

ga_hits = decode_and_filter(ga_top)
bo_hits = decode_and_filter(bo_top)
rand_hits = decode_and_filter(rand_top)

print(f"GA     post-filter: {len(ga_hits)}/50")
print(f"Optuna post-filter: {len(bo_hits)}/50")
print(f"Random post-filter: {len(rand_hits)}/50")

print("\nFirst 5 valid GA hits:")
for m in ga_hits[:5]: print(" ", m)
print("\nFirst 5 valid Optuna hits:")
for m in bo_hits[:5]: print(" ", m)

---

## Beyond this notebook

- **Multi-objective** — `score` returns a scalar here. For Pareto search
  swap PyGAD for DEAP NSGA-II (template in
  `research/experiments/inverse_qsar/ga_new.ipynb`) or use Optuna's
  `NSGAIISampler` with a multi-output `objective`.
- **Scaffold constraints** — pre-seed must-have fragment indices in
  `sample_candidate` (Optuna) or fix them in PyGAD's `gene_space`.
- **Continuous Bayesian optimization (BoTorch)** — the encoder returns a
  per-atom continuous tensor `atoms_vectors ∈ ℝ^{B×51×512}`, and the decoder
  module accepts it directly via `VQGAE.decoder.forward(...)`. The public
  `VQGAE.decode` API quantizes through the codebook first, so you don't get
  a continuous round-trip out of the box. Adding a small helper:

  ```python
  # in VQGAE/models.py
  def decode_from_atoms_vectors(self, atoms_vectors, sizes):
      p_atoms, p_bonds = self.decoder(atoms_vectors)
      atoms = torch.argmax(p_atoms, -1)
      bonds = torch.argmax(p_bonds, -3)
      return atoms, bonds, sizes
  ```

  unlocks BO over the continuous latent (or the 512-dim molecule-level
  `feature_vector` if you also add a small re-projection head).
